# Smoke test hạ tầng ĐA1 trên Colab

Đo bge-m3, bge-reranker-v2-m3, Qdrant hybrid và Qwen2.5 7B (Ollama) trên GPU Colab, cùng script `scripts/smoke_test.py` đã chạy trên laptop.

1. **Runtime → Change runtime type → chọn GPU (L4 hoặc T4) → Save → Connect.**
2. Ghi số compute unit còn lại và unit/giờ ở bảng **Resources**.
3. **Runtime → Run all.** Cell cuối hỏi xác nhận rồi tắt runtime để dừng tính unit.
4. Ghi lại số unit còn lại sau khi chạy.

Chỉ dùng dữ liệu giả tích hợp trong script; không đưa dữ liệu thật vào notebook này.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone repo (public)

In [ ]:
!git clone -q https://github.com/daisyngocha/bv-rag.git /content/bv-rag
%cd /content/bv-rag
!git log --oneline -1

## 3. Cài thư viện
Torch bản CUDA có sẵn trên Colab. Nếu pip báo cần khởi động lại: Runtime → Restart session, rồi chạy tiếp từ cell 2.

In [ ]:
!pip -q install -r requirements/base.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|flagembedding|qdrant-client) "

## 4. Cài Ollama
`pciutils`, `lshw` phải có trước để Ollama nhận GPU.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 5. Bật Ollama và tải model
Dòng cuối phải có `cuda`.

In [ ]:
import subprocess, time, requests
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); break
    except requests.RequestException:
        time.sleep(1)
!ollama pull qwen2.5:7b-instruct-q4_K_M
!grep -iE "inference compute|library=cuda" /content/ollama.log | head -3

## 6. Chạy smoke test

In [ ]:
!python scripts/smoke_test.py --qdrant-url :memory:

## 7. Xác nhận Ollama chạy trên GPU
Chạy ngay sau cell 6. Cột PROCESSOR phải là `100% GPU`.

In [ ]:
!ollama ps

## 8. Ghi thông tin môi trường, đổi tên file, tải về

In [ ]:
import glob, json, subprocess, datetime, requests
from google.colab import files

src = sorted(glob.glob("logs/smoke_*.json"))[-1]
d = json.load(open(src, encoding="utf-8"))
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().split()[-1]
tags = requests.get("http://localhost:11434/api/tags").json()["models"]
d["colab"] = {
    "gpu": gpu,
    "ollama_version": requests.get("http://localhost:11434/api/version").json()["version"],
    "model_digest": next(m["digest"] for m in tags if m["name"].startswith("qwen2.5:7b-instruct-q4_K_M")),
    "packages": subprocess.check_output("pip list 2>/dev/null | grep -iE '^(torch|transformers|flagembedding|qdrant-client) '",
                                        shell=True).decode().strip().split("\n"),
    "repo_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip(),
}
dst = f"docs/smoke_colab_{gpu}_{datetime.date.today():%Y%m%d}.json"
json.dump(d, open(dst, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(dst)
files.download(dst)

## 9. Tắt runtime
Runtime vẫn tính unit khi không chạy gì. Chỉ trả lời `y` sau khi file JSON đã tải xong.

In [ ]:
from google.colab import runtime
if input("Đã tải file JSON về máy? (y/n) ").strip().lower() == "y":
    runtime.unassign()